# Warp Drape Viewer

Interactive inspector for GarmentCodeData drape cases. Run the cells top to bottom; the final cell opens a Warp `OpenGLRenderer` window.

Controls (inside the viewer window):
- `Left` / `Right`: previous / next frame
- `Home` / `End`: first / last frame
- `B`: toggle body overlay
- `R`: toggle reference drape overlay
- `S`: toggle simulated drape overlay
- `I`: toggle initial (box mesh) overlay
- `Space`: pause, `Esc`: close

> The dataset root and body path come from `QYDP_GCD_ROOT` / `QYDP_GCD_BODY` (machine-specific values live in `LOCAL_DEV.md`, gitignored). A CUDA-capable Warp runtime is required (no non-GPU fallback).

In [2]:
"""Setup: locate the module, dataset, and helpers."""
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in (REPO_ROOT, REPO_ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
from conftest import _resolve_qydp
from harness.gcd.loader import load_element

qydp = _resolve_qydp()
print("Qianyi_DP", getattr(qydp, "__version__", "dev"), "loaded")

def data_root() -> Path:
    value = os.environ.get("QYDP_GCD_ROOT")
    if not value:
        raise RuntimeError("QYDP_GCD_ROOT is not set (see LOCAL_DEV.md)")
    return Path(value)

def body_obj() -> Path | None:
    value = os.environ.get("QYDP_GCD_BODY")
    return Path(value) if value else None

def load_frames(artifact_dir: Path) -> dict:
    data = np.load(artifact_dir / "frames.npz")
    return {"local": data["local"], "world": data["world"], "timestamps": data["timestamps"]}


Qianyi_DP dev loaded


In [3]:
"""Select a case: either a specific element id + artifact dir, or the first
failing case of a batch summary (see the batch cell below)."""

def select_case(element_id: str | None = None, artifact_dir: Path | None = None):
    root = data_root()
    element_id = element_id or sorted(d.name for d in root.iterdir() if d.is_dir())[0]
    element_dir = root / element_id
    artifact_dir = artifact_dir or (REPO_ROOT / "tests" / "artifacts" / element_id)
    element = load_element(element_dir, body_obj())
    has_frames = (artifact_dir / "frames.npz").is_file()
    if not has_frames:
        print(f"no frames.npz in {artifact_dir}; sim overlay disabled (run the case first)")
    frames = load_frames(artifact_dir) if has_frames else None
    return element, frames, artifact_dir

In [4]:
"""Build renderer geometry for the selected case.

Returns a dict of meshes; each mesh has points/indices/colors in metres (Z-up).
"""
def build_scene(element):
    cloth = [m for m in element.input_data["mesh_list"] if m["object_type"] == 0]
    offsets = []
    total = 0
    for m in cloth:
        offsets.append(total)
        total += len(m["vertices"]) // 3
    scene = {
        "cloth": [
            {
                "name": f"panel_{i}",
                "points": np.asarray(m["vertices"], dtype=np.float32).reshape(-1, 3),
                "indices": np.asarray(m["triangles"], dtype=np.int32).reshape(-1),
                "offset": offsets[i],
            }
            for i, m in enumerate(cloth)
        ],
        "body": None,
        "reference": None,
    }
    for m in element.input_data["mesh_list"]:
        if m["object_type"] == 1:
            scene["body"] = {
                "points": np.asarray(m["vertices"], dtype=np.float32).reshape(-1, 3),
                "indices": np.asarray(m["triangles"], dtype=np.int32).reshape(-1),
            }
    if element.reference_vertices is not None and element.reference_faces is not None:
        scene["reference"] = {
            "points": np.asarray(element.reference_vertices, dtype=np.float32),
            "indices": np.asarray(element.reference_faces, dtype=np.int32).reshape(-1),
        }
    return scene

In [ ]:
"""Interactive Warp viewer (CUDA-capable runtime required).

Frame scrubbing and overlay toggles use ``register_key_press_callback``;
the rendering loop follows the ``t3bvh_test.ipynb`` pattern.
"""
import time
import traceback
import warp as wp
import warp.render
import pyglet

wp.init()

def run_viewer(element, frames, fps=30):
    scene = build_scene(element)
    n_frames = frames["local"].shape[0] if frames is not None else 1

    renderer = warp.render.OpenGLRenderer(
        title="Warp Drape Viewer",
        screen_width=1280,
        screen_height=900,
        up_axis="Z",
        camera_pos=(2.2, 2.6, 2.2),
        camera_front=(-0.6, -0.5, -0.6),
        near_plane=0.01,
        far_plane=10.0,
        vsync=True,
    )

    state = {"frame": 0, "show": {"sim": True, "initial": True, "reference": True, "body": True}}

    def on_key(symbol, modifiers):
        key = pyglet.window.key
        if symbol == key.LEFT:
            state["frame"] = max(0, state["frame"] - 1)
        elif symbol == key.RIGHT:
            state["frame"] = min(n_frames - 1, state["frame"] + 1)
        elif symbol == key.HOME:
            state["frame"] = 0
        elif symbol == key.END:
            state["frame"] = n_frames - 1
        elif symbol == key.B:
            state["show"]["body"] = not state["show"]["body"]
        elif symbol == key.R:
            state["show"]["reference"] = not state["show"]["reference"]
        elif symbol == key.S:
            state["show"]["sim"] = not state["show"]["sim"]
        elif symbol == key.I:
            state["show"]["initial"] = not state["show"]["initial"]
        return None

    renderer.register_key_press_callback(on_key)
    t = 0.0
    dt = 1.0 / fps
    try:
        while not renderer.window.has_exit:
            renderer.begin_frame(t)
            frame = state["frame"]
            if state["show"]["initial"]:
                for mesh in scene["cloth"]:
                    renderer.render_mesh(
                        name="initial_" + mesh["name"],
                        points=mesh["points"],
                        indices=mesh["indices"],
                        colors=(0.9, 0.9, 0.9),
                    )
            if state["show"]["sim"] and frames is not None:
                points = frames["local"][frame].astype(np.float32)
                for mesh in scene["cloth"]:
                    renderer.render_mesh(
                        name="sim_" + mesh["name"],
                        points=points[mesh["offset"]:mesh["offset"] + len(mesh["points"])],
                        indices=mesh["indices"],
                        colors=(0.15, 0.55, 0.95),
                    )
            if state["show"]["reference"] and scene["reference"] is not None:
                renderer.render_mesh(
                    name="reference",
                    points=scene["reference"]["points"],
                    indices=scene["reference"]["indices"],
                    colors=(0.9, 0.4, 0.1),
                )
            if state["show"]["body"] and scene["body"] is not None:
                renderer.render_mesh(
                    name="body",
                    points=scene["body"]["points"],
                    indices=scene["body"]["indices"],
                    colors=(0.75, 0.75, 0.55),
                )
            renderer.end_frame()
            t += dt
            time.sleep(dt / 2.0)
    except Exception as exc:
        traceback.print_exc()
        print("viewer stopped:", exc)
    finally:
        try:
            renderer.close()
        except Exception as close_exc:
            print("renderer close failed:", close_exc)

# Open the selected case. Set element_id to inspect a specific element.
# QYDP_GCD_ROOT / QYDP_GCD_BODY must be set in the environment (see LOCAL_DEV.md).
element, frames, artifact_dir = select_case(element_id="rand_00YONAPXZE")
run_viewer(element, frames)

## Batch results integration

Open a batch summary to list failures and jump straight into a failing case.

In [ ]:
"""List failing cases from a batch summary and open one in the viewer."""
import json

def load_batch_summary(summary_path: str | Path):
    path = Path(summary_path)
    summary = json.loads(path.read_text(encoding="utf-8"))
    failures = [c for c in summary["cases"] if c["status"] == "failed"]
    print(f"batch: {summary['passed']} passed, {summary['failed']} failed")
    for c in failures:
        print(" -", c["id"], c.get("failure_class"))
    return summary, failures

# summary, failures = load_batch_summary("tests/artifacts/<batch>/batch_summary.json")
# if failures:
#     element, frames, artifact_dir = select_case(element_id=failures[0]["id"])
#     run_viewer(element, frames)